# Branch 1 — Numerical Models

Uses the **same SEC numerical dataset, target construction, neutral band, feature engineering, and chronological split logic as v6.1**. Models: L2 Logistic Regression, Balanced L2 Logistic Regression, Random Forest, XGBoost, and MLP neural network.

In [ ]:
!pip -q install pyarrow openpyxl xgboost sentence-transformers transformers accelerate beautifulsoup4 requests tqdm

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH: Path | None = None
NEUTRAL_BAND = 0.02
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)

OUTPUT_DIR = Path('/content/semiconductor_branch_numerical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load the same dataset used by v6.1

In [ ]:
def discover_data_path() -> Path:
    candidates = [
        Path('/content/drive/MyDrive/sec_research_semiconductor/semiconductor_sec_numeric_text.parquet'),
        Path('/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet'),
        Path('/content/semiconductor_sec_numeric_text.parquet'),
        Path('/content/sec_experiment_semiconductor.parquet'),
        Path('semiconductor_sec_numeric_text.parquet'),
        Path('semiconductor_sec_numeric_text.csv'),
        Path('sec_experiment_semiconductor.parquet'),
        Path('model_dataset.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            print('Found dataset automatically:', candidate)
            return candidate

    try:
        from google.colab import files
        print('Upload the same parquet/CSV dataset used by v6.1.')
        uploaded = files.upload()
        choices = [Path(name) for name in uploaded if Path(name).suffix.lower() in {'.parquet', '.csv'}]
        if not choices:
            raise FileNotFoundError('Upload a .parquet or .csv file.')
        return choices[0]
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_PATH to the v6.1 dataset path.') from exc

resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()
if resolved_path.suffix.lower() == '.parquet':
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == '.csv':
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

print('Loaded:', resolved_path)
print('Rows:', len(raw), '| Columns:', len(raw.columns))

## Rebuild target exactly as v6.1

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

## Financial feature engineering

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

## Chronological train/validation/test split

In [ ]:
# Use an existing split only when it contains all three required groups.
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "The existing split column does not contain all three groups. "
            "Rebuilding the split chronologically."
        )

    data["split"] = pd.NA

    # Use the most conservative available timestamp:
    # when the target became knowable, then the feature cutoff, then quarter end.
    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required to create "
            "train, validation, and test splits."
        )

    # Automatic chronological 60% / 20% / 20% split by distinct dates.
    train_position = max(0, min(len(eligible_dates) - 3, int(len(eligible_dates) * 0.60) - 1))
    validation_position = max(
        train_position + 1,
        min(len(eligible_dates) - 2, int(len(eligible_dates) * 0.80) - 1),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)

model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()
model_data["target_clean"] = model_data["target_clean"].astype(int)

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Could not create these splits: {sorted(missing_splits)}. "
        "The dataset may have too few labeled dates after applying the "
        "neutral band."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[model_data["split"] == split_name]
    if split_frame["target_clean"].nunique() < 2:
        print(
            f"Warning: {split_name} contains only one target class after "
            f"applying the {NEUTRAL_BAND:.1%} neutral band. "
            "Try reducing NEUTRAL_BAND to 0.01 if model fitting fails."
        )

## Numerical feature list

In [ ]:
candidate_numeric_features = [
    'revenue_yoy_growth', 'revenue_momentum', 'sequential_revenue_growth',
    'gross_margin', 'gross_margin_change', 'operating_margin',
    'operating_margin_change', 'cash_flow_margin', 'cash_flow_margin_change',
    'inventory_to_ttm_revenue', 'inventory_to_ttm_revenue_change',
    'inventory_yoy_growth', 'inventory_revenue_growth_gap',
    'receivables_to_ttm_revenue', 'receivables_to_ttm_revenue_change',
    'receivables_yoy_growth', 'receivables_revenue_growth_gap',
    'capex_to_revenue', 'capex_to_revenue_change', 'capex_yoy_growth',
    'log_assets', 'liabilities_to_assets',
]

candidate_numeric_features += [
    f'{feature}_relative_to_sector' for feature in relative_base_features
]

numeric_features = [
    c for c in dict.fromkeys(candidate_numeric_features)
    if c in model_data.columns
    and model_data[c].notna().sum() >= 10
    and model_data[c].nunique(dropna=True) > 1
]

categorical_features = [
    c for c in ['fiscal_quarter', 'semiconductor_subgroup']
    if c in model_data.columns and model_data[c].notna().any()
]
feature_columns = numeric_features + categorical_features
print('Numerical features:', len(numeric_features))
print('Categorical features:', categorical_features)
display(pd.DataFrame({'feature': feature_columns}))

## Train and compare all numerical models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

OUTPUT_DIR = Path('/content/semiconductor_branch_numerical')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_preprocessor():
    transformers = []
    if numeric_features:
        transformers.append(('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features))
    if categorical_features:
        transformers.append(('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features))
    return ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)


def choose_threshold(y_true, probabilities):
    scores = [balanced_accuracy_score(y_true, (probabilities >= t).astype(int)) for t in THRESHOLD_GRID]
    i = int(np.argmax(scores))
    return float(THRESHOLD_GRID[i]), float(scores[i])


def evaluate_row(model_name, y_true, probabilities, predictions, baseline_probability):
    row = {
        'model': model_name,
        'rows': len(y_true),
        'accuracy': accuracy_score(y_true, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
        'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0),
        'acceleration_precision': precision_score(y_true, predictions, pos_label=1, zero_division=0),
        'acceleration_recall': recall_score(y_true, predictions, pos_label=1, zero_division=0),
        'deceleration_recall': recall_score(y_true, predictions, pos_label=0, zero_division=0),
        'brier_score': brier_score_loss(y_true, probabilities),
        'log_loss': log_loss(y_true, np.c_[1-probabilities, probabilities], labels=[0,1]),
    }
    base = np.full(len(y_true), baseline_probability)
    base_brier = brier_score_loss(y_true, base)
    row['brier_skill_score'] = 1 - row['brier_score'] / base_brier if base_brier > 0 else np.nan
    row['roc_auc'] = roc_auc_score(y_true, probabilities) if pd.Series(y_true).nunique() == 2 else np.nan
    row['average_precision'] = average_precision_score(y_true, probabilities) if pd.Series(y_true).nunique() == 2 else np.nan
    return row

train = model_data[model_data.split == 'train'].copy()
validation = model_data[model_data.split == 'validation'].copy()
test = model_data[model_data.split == 'test'].copy()
train_validation = pd.concat([train, validation], ignore_index=True)
y_train, y_val, y_test = train.target_clean, validation.target_clean, test.target_clean
baseline_probability = float(train_validation.target_clean.mean())

model_candidates = []

# 1) L2 Logistic Regression
for C in C_GRID:
    model_candidates.append(('L2 Logistic Regression', {'C': C}, Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', LogisticRegression(C=C, penalty='l2', solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE)),
    ])))

# 2) Balanced L2 Logistic Regression
for C in C_GRID:
    model_candidates.append(('Balanced L2 Logistic Regression', {'C': C}, Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', LogisticRegression(C=C, penalty='l2', class_weight='balanced', solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE)),
    ])))

# 3) Random Forest
for params in [
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 500, 'max_depth': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 500, 'max_depth': 10, 'min_samples_leaf': 3, 'max_features': 0.7},
]:
    model_candidates.append(('Random Forest', params, Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', RandomForestClassifier(**params, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)),
    ])))

# 4) XGBoost
positive = max(1, int((y_train == 1).sum()))
negative = max(1, int((y_train == 0).sum()))
scale_pos_weight = negative / positive
for params in [
    {'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.03, 'subsample': 0.9, 'colsample_bytree': 0.9},
    {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.03, 'subsample': 0.9, 'colsample_bytree': 0.9},
    {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.8},
]:
    full = dict(params, scale_pos_weight=scale_pos_weight)
    model_candidates.append(('XGBoost', full, Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', XGBClassifier(**full, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)),
    ])))

# 5) MLP neural network
for params in [
    {'hidden_layer_sizes': (32,), 'alpha': 0.001},
    {'hidden_layer_sizes': (64, 32), 'alpha': 0.001},
    {'hidden_layer_sizes': (64, 32), 'alpha': 0.01},
]:
    model_candidates.append(('MLP Neural Network', params, Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MLPClassifier(**params, activation='relu', solver='adam', max_iter=1500, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE)),
    ])))

best_by_model = {}
selection_rows = []
for name, params, pipe in model_candidates:
    pipe.fit(train[feature_columns], y_train)
    p = pipe.predict_proba(validation[feature_columns])[:,1]
    brier = brier_score_loss(y_val, p)
    threshold, val_bal = choose_threshold(y_val, p)
    selection_rows.append({'model': name, 'params': str(params), 'validation_brier': brier, 'threshold': threshold, 'validation_balanced_accuracy': val_bal})
    if name not in best_by_model or brier < best_by_model[name]['brier']:
        best_by_model[name] = {'brier': brier, 'params': params, 'threshold': threshold, 'pipeline': pipe}

results = []
predictions = test[[c for c in ['cik','ticker','company_name','quarter_end','future_growth_change','target_clean'] if c in test.columns]].copy()
final_models = {}

# Baselines
prior_p = np.full(len(test), baseline_probability)
prior_pred = (prior_p >= 0.5).astype(int)
results.append(evaluate_row('Prior-probability baseline', y_test, prior_p, prior_pred, baseline_probability))
majority_class = int(train_validation.target_clean.mode().iloc[0])
maj_p = np.full(len(test), float(majority_class))
maj_pred = np.full(len(test), majority_class)
results.append(evaluate_row('Majority-class baseline', y_test, maj_p, maj_pred, baseline_probability))

for name, chosen in best_by_model.items():
    params = chosen['params']
    # rebuild selected model cleanly for train+validation
    if name == 'L2 Logistic Regression':
        clf = LogisticRegression(C=params['C'], penalty='l2', solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE)
    elif name == 'Balanced L2 Logistic Regression':
        clf = LogisticRegression(C=params['C'], penalty='l2', class_weight='balanced', solver='lbfgs', max_iter=5000, random_state=RANDOM_STATE)
    elif name == 'Random Forest':
        clf = RandomForestClassifier(**params, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
    elif name == 'XGBoost':
        clf = XGBClassifier(**params, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
    else:
        clf = MLPClassifier(**params, activation='relu', solver='adam', max_iter=1500, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE)
    pipe = Pipeline([('preprocessor', make_preprocessor()), ('classifier', clf)])
    pipe.fit(train_validation[feature_columns], train_validation.target_clean)
    p = pipe.predict_proba(test[feature_columns])[:,1]
    pred = (p >= chosen['threshold']).astype(int)
    row = evaluate_row(name, y_test, p, pred, baseline_probability)
    row.update({'selected_params': str(params), 'threshold': chosen['threshold'], 'validation_brier': chosen['brier']})
    results.append(row)
    predictions[f'{name}_probability'] = p
    predictions[f'{name}_prediction'] = pred
    final_models[name] = pipe

selection_table = pd.DataFrame(selection_rows).sort_values(['model','validation_brier'])
results_table = pd.DataFrame(results).sort_values(['brier_score','balanced_accuracy'], ascending=[True,False])
display(results_table)

selection_table.to_csv(OUTPUT_DIR/'numerical_validation_selection.csv', index=False)
results_table.to_csv(OUTPUT_DIR/'numerical_test_results.csv', index=False)
predictions.to_csv(OUTPUT_DIR/'numerical_test_predictions.csv', index=False)
with pd.ExcelWriter(OUTPUT_DIR/'numerical_branch_results.xlsx', engine='openpyxl') as writer:
    results_table.to_excel(writer, sheet_name='Model_Results', index=False)
    selection_table.to_excel(writer, sheet_name='Validation_Selection', index=False)
    predictions.to_excel(writer, sheet_name='Test_Predictions', index=False)
joblib.dump(final_models, OUTPUT_DIR/'numerical_final_models.joblib')
print('Saved to:', OUTPUT_DIR)